
## `Лабораторная работа №3 "Задачи NLP"`



#### Фамилия, имя:

Дата выдачи: <span style="color:red">__07 апреля__</span>.

Срок сдачи: <span style="color:red">__21 апреля__</span>.

Формат сдачи: отчёт и ссылка на блокнот/файл с кодом.

Максимальная оценка: __10 баллов__




## `Задание 1. Классификация текста с помощью RNN (5 баллов)`



[Данные](https://drive.google.com/drive/folders/1epWf0Wwixxv9juicjdCYYVz351vEe9yM?usp=drive_link) представляют собой тексты разных авторов. Обучающий набор содержит текст - его автора (метку к нему), всего 8 авторов. Тестовый набор содержит только тексты.

Необходимо решить задачу многоклассовой классификации текстов по авторам.

In [ ]:
# загрузим данные
!gdown --folder https://drive.google.com/drive/folders/1epWf0Wwixxv9juicjdCYYVz351vEe9yM?usp=drive_link

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
import nltk
from tqdm import tqdm_notebook
import torch.nn as nn
import torch.nn.functional as F

from collections import Counter
from typing import List
import string

import seaborn
seaborn.set(palette='summer')
nltk.download('all')

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

### Подготовка данных

Загрузим датасет и посмотрим на данные:

In [ ]:
import pandas as pd
train_data = pd.read_csv('/content/Text_Author/train_authors.csv')
train_data.head(1)

In [ ]:
test_data = pd.read_csv('/content/Text_Author/test_authors.csv')
test_data.head()

In [ ]:
# Назначим каждому автору идентификатор
writers = ['Akunin', 'Bulychev', 'Chehov', 'Dostoevsky', 'Gogol', 'King',
          'Pratchett', 'Remark']
writers_to_label = {writer: i for i, writer in enumerate(writers)}
label_to_writers = {i: writer for i, writer in enumerate(writers)}

Подготовим данные для задачи классификации текстов по авторам, преобразуя сырые данные в словарь dataset['train'] и dataset['test'].


In [ ]:
dataset = {}

dataset['train'] = [{'text':text, 'author':writers_to_label[label]} \
              for text, label in zip(np.array(train_data['text']), np.array(train_data['author']))]
dataset['test'] = [{'text':text, 'author': 0} \
              for text in np.array(test_data['text'])]

Можно сделать иначе используя класс `Dataset`.


In [ ]:
from torch.utils.data import Dataset

class TextDataset(Dataset):
    def __init__(self, texts, authors):
        self.texts = texts
        self.authors = authors

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {'text': self.texts[idx], 'author': self.authors[idx]}

**1)** Заведите Dataloader для загрузки данных нейронную сеть.

**2)** Разработайте и обучите RNN  на тренировочных данных.

Вкачестве рекуррентных слоёв можно выбрать:
- базовый RNN,
- LSTM,
- GRU,
- Применение двунаправленных рекуррентных слоев (Bidirectional RNN/LSTM/GRU).

**3)** Проведите эксперитенты с:
- Увеличением глубины модели. Используйте параметр `num_layers` для создания многослойной RNN.
При этом выходы каждого предыдущего слоя передаются в качестве входов следующему.

- Агрегацией выходных последовательностей. Помимо стандартных подходов (max, mean, last), рассмотрите, например, конкатенацию различных видов агрегации, использование различных пулинговых операций, или Ваш вариант.

**4)** Сделайте выводы и выберите модель с лучшей accuracy.

### Инференс модели

Ниже функция для оценки точности модели на данных из DataLoader.

Здесь model — ваша обученная модель, dataloader — test_dataloader, построенный на основе тестовой части данных (dataset['test']).

In [ ]:

all_words = []
for text in train_data['text']:
    tokens = word_tokenize(str(text).lower())
    all_words.extend(tokens)


max_words = 10000
word_counts = Counter(all_words)
common_words = [word for word, _ in word_counts.most_common(max_words)]


word2idx = {'<PAD>': 0, '<UNK>': 1}
for idx, word in enumerate(common_words, start=2):
    word2idx[word] = idx


def text_to_indices(text):
    tokens = word_tokenize(str(text).lower())
    return [word2idx.get(word, word2idx['<UNK>']) for word in tokens]


train_texts_idx = [text_to_indices(t) for t in train_data['text']]
train_labels = [writers_to_label[l] for l in train_data['author']]

In [ ]:
from torch.nn.utils.rnn import pad_sequence


train_dataset = TextDataset(train_texts_idx, train_labels)

# Функция для сборки батча и выравнивания текстов
def collate_fn(batch):

    texts = [torch.tensor(item['text']) for item in batch]
    authors = [item['author'] for item in batch]


    padded_texts = pad_sequence(texts, batch_first=True, padding_value=0)
    authors_tensor = torch.tensor(authors, dtype=torch.long)


    return {'input_ids': padded_texts, 'labels': authors_tensor}


train_dataloader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)


batch = next(iter(train_dataloader))
print(f"Размерность батча с текстами: {batch['input_ids'].shape}")
print(f"Размерность батча с ответами: {batch['labels'].shape}")

In [ ]:
def get_predictions(model, dataloader):

    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in tqdm_notebook(dataloader,
                                   desc=f'Evaluating'):

            logits = model(batch['input_ids'])

            predictions.append(logits.argmax(dim=1))

    predictions = torch.cat(predictions).data.cpu().numpy()

    return predictions

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes, num_layers=1, rnn_type='LSTM'):
        super(TextClassifier, self).__init__()


        self.embedding = nn.Embedding(num_embeddings=vocab_size,
                                      embedding_dim=embedding_dim,
                                      padding_idx=0)


        self.rnn_type = rnn_type
        if rnn_type == 'LSTM':
            self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        elif rnn_type == 'GRU':
            self.rnn = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        else:
            self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)


        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):

        embedded = self.embedding(x)

        if self.rnn_type == 'LSTM':

            rnn_out, (hidden, cell) = self.rnn(embedded)
        else:

            rnn_out, hidden = self.rnn(embedded)


        last_hidden = hidden[-1, :, :]


        logits = self.fc(last_hidden)

        return logits

In [ ]:

VOCAB_SIZE = len(word2idx)
EMBEDDING_DIM = 128
HIDDEN_DIM = 256
NUM_CLASSES = 8
NUM_LAYERS = 2
EPOCHS = 10

# Инициализируем модель, переносим на GPU
model = TextClassifier(vocab_size=VOCAB_SIZE,
                       embedding_dim=EMBEDDING_DIM,
                       hidden_dim=HIDDEN_DIM,
                       num_classes=NUM_CLASSES,
                       num_layers=NUM_LAYERS,
                       rnn_type='LSTM').to(device)


criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct_predictions = 0
    total_samples = 0

    for batch in train_dataloader:

        inputs = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)


        optimizer.zero_grad()

        logits = model(inputs)


        loss = criterion(logits, labels)


        loss.backward()


        optimizer.step()


        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct_predictions += (preds == labels).sum().item()
        total_samples += labels.size(0)

    avg_loss = total_loss / len(train_dataloader)
    accuracy = correct_predictions / total_samples
    print(f"Эпоха {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Точность: {accuracy:.4f}")

In [ ]:

test_texts_idx = [text_to_indices(t) for t in test_data['text']]

test_labels = [0] * len(test_data)

test_dataset = TextDataset(test_texts_idx, test_labels)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

In [ ]:
from tqdm.notebook import tqdm

def get_predictions(model, dataloader):
    """
    Calculate accuracy on data from dataloader.
    """
    model.eval()
    predictions = []
    with torch.no_grad():

        for batch in tqdm(dataloader, desc='Evaluating'):


            input_ids = batch['input_ids'].to(device)


            logits = model(input_ids)


            predictions.append(logits.argmax(dim=1))

    predictions = torch.cat(predictions).data.cpu().numpy()

    return predictions

In [ ]:

test_texts_idx = [text_to_indices(t) for t in test_data['text']]


test_labels = [0] * len(test_data)


test_dataset = TextDataset(test_texts_idx, test_labels)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)


predictions_idx = get_predictions(model, test_dataloader)


predictions = [label_to_writers[x] for x in predictions_idx]



### Сохраните результат

In [ ]:
np.save('submission_rnn03.npy', predictions, allow_pickle=True)
print('Ответ сохранен в файл `submission_rnn03.npy`')

## `Задание 2. Классификация текста с помощью предобученной языковой модели BERT (3 балла)`

Рассматривается [датасет](https://drive.google.com/drive/folders/151O9W-yp6TMP0Wmrm_wzbu8F7sq4cm_4?usp=sharing)
**SST-2** (Stanford Sentiment Treebank версии 2).


In [ ]:
import os
import random

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt
from IPython.display import clear_output

### Загрузка данных


`holdout_texts.npy` - файл содержит holdout выборку.

holdout выборка - это отдельная часть неразмеченных данных, которая НЕ используется при обучении модели и служит исключительно для финальной оценки её качества после завершения всех экспериментов.

`STT2_train_task.tsv` - файл с обучающей выборкой.


In [ ]:
# загрузим данные
!gdown --folder https://drive.google.com/drive/folders/151O9W-yp6TMP0Wmrm_wzbu8F7sq4cm_4?usp=sharing

In [ ]:
# преобразуем holdout-выборку в numpy.array
texts_holdout = np.load('/content/SST2/holdout_texts.npy', allow_pickle=True)
texts_holdout[:5]


In [ ]:
# Загрузим обучающие и тестовые размеченные данные
df = pd.read_csv(
    '/content/SST2/STT2_train_task.tsv.txt',
    delimiter='\t',
    header=None
)


In [ ]:
df.info()

In [ ]:
texts_train = df[0].values[:5000]
y_train = df[1].values[:5000]
texts_test = df[0].values[5000:]
y_test = df[1].values[5000:]


**1)** Напишите функцию получения эмбеддингов.

Сначала необходимо получить эмбендинги предложений с помощью обученной `BERT':

- Загрузить предобученный токенизатор и модель.

- Токенизировать текст с добавлением специальных токенов и преобразовать в тензоры.

Функция
`tokenizer.encode(text, add_special_tokens=True)`
возвращает ID токенов.

- Пропустить через модель и получить выходы.

- Извлечь эмбеддинг для всего предложения (возьмем скрытое состояние, соответствующее [CLS] токену).



In [ ]:
!pip install transformers -q

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
from tqdm.auto import tqdm


model_name = 'bert-base-uncased'


tokenizer = AutoTokenizer.from_pretrained(model_name)


bert_model = AutoModel.from_pretrained(model_name).to(device)

In [ ]:
def get_bert_embeddings(texts, model, tokenizer, batch_size=32, pooling_strategy='cls'):
    """
    Получение эмбеддингов текста с помощью BERT
    """
    model.eval()
    all_embeddings = []


    for i in tqdm(range(0, len(texts), batch_size), desc="Получение эмбеддингов"):
        batch_texts = list(texts[i : i + batch_size])


        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        )


        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)

        with torch.no_grad():
            outputs = model(input_ids, attention_mask=attention_mask)

        if pooling_strategy == 'cls':

            embeddings = outputs.last_hidden_state[:, 0, :]

        elif pooling_strategy == 'mean':
            =
            embeddings = outputs.last_hidden_state.mean(dim=1)


        all_embeddings.append(embeddings.cpu().numpy())

  =
    return np.vstack(all_embeddings)

In [ ]:
print("Обрабатываем обучающую выборку...")
X_train_bert = get_bert_embeddings(texts_train, bert_model, tokenizer)

print("Обрабатываем тестовую выборку...")
X_test_bert = get_bert_embeddings(texts_test, bert_model, tokenizer)

print("Обрабатываем holdout выборку...")
X_holdout_bert = get_bert_embeddings(texts_holdout, bert_model, tokenizer)

print(f"Размерность признаков train: {X_train_bert.shape}")

**2)** Обучите логистическую регрессию

Датасет для обучения логистической регрессии состоит из векторов, полученных из предобученной модели BERT. В качестве признаков используются эмбеддинги токена [CLS], которые извлекаются из выходного слоя трансформерного блока модели.
Каждая строка датасета соответствует отдельному предложению, а столбцы представляют активации скрытых нейронов полносвязного слоя, следующего за трансформерной архитектурой. Эти векторные представления служат входными признаками для классификатора.

Ваша модель должна давать точность > 80%

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

=
lr_model = LogisticRegression(max_iter=1000, random_state=42)

=
print("Обучаем логистическую регрессию...")
lr_model.fit(X_train_bert, y_train)


test_preds = lr_model.predict(X_test_bert)
accuracy = accuracy_score(y_test, test_preds)
print(f"Точность на тестовой выборке: {accuracy:.4f}")


probs_train = lr_model.predict_proba(X_train_bert)
probs_test = lr_model.predict_proba(X_test_bert)
probs_holdout = lr_model.predict_proba(X_holdout_bert)



Сохраните в словарь `out_dict` вероятности принадлежности к нулевому и первому классу соответственно, для кадой выборки:

In [ ]:
out_dict = {
    'train': probs_train,
    'test': probs_test,
    'holdout': probs_holdout
}

### Сохраните результат

In [ ]:
np.save('submission_bert03.npy', out_dict, allow_pickle=True)
print('Ответ сохранен в файл `submission_bert03.npy`')

## `Задание 3. Реализовать механизм внимания (2 балла)`


Рассмотрим архитектуру энкодера-декодера, где энкодер преобразует входную последовательность в набор скрытых состояний $h_{e1}, ..., h_{eN}$, а декодер генерирует выходную последовательность, используя свои состояния $h_{dt}$ в каждый момент времени $t$.


**Вектор внимания** $a_{dt}$ в момент времени $t$ представляет собой распределение важности по всем состояниям энкодера:


$$a_{dt} = [a_{i,t},...,a_{N, t}]$$


где каждый элемент $a_{i, t} = f(h_{ei}, h_{dt})$ вычисляется с помощью функции внимания $f$, оценивающей релевантность $i$-го состояния энкодера текущему состоянию декодера.

**Типы функций внимания**:

1. **Внимание на основе скалярного произведения (Dot-Product Attention)**
Самый простой и вычислительно эффективный подход, непосредственно измеряющий сходство между векторами:

$$f(h_{ei}, h_{dt}) = h_{dt}^Th_{ei}$$

Преимущества:

- Минимальные вычислительные затраты.

- Не требует дополнительных обучаемых параметров.

2. **Мультипликативное внимание (Multiplicative Attention)**

Вводит обучаемую матрицу весов $W$ для моделирования взаимодействий между пространствами энкодера и декодера:

$$f(h_{ei}, h_{dt}) = h_{dt}^T*W*h_{ei}$$

Преимущества:

- Позволяет моделировать более сложные взаимосвязи.

- Матрица $W$ адаптируется в процессе обучения.

- Эффективнее скалярного произведения при разных размерностях векторов.

3. **Аддитивное внимание (Additive Attention)**
Использует двухслойную нейронную сеть с нелинейной активацией (например, tanh) для вычисления оценки внимания:

$$f(h_{ei}, h_{dt}) = v^T*tanh(W_1*h_{ei}+W_2*h_{dt})$$

где $W_1$, $W_2$ — матрицы весов, а $v$ — вектор весов для получения скалярного значения.

Преимущества:

- Более выразительная модель за счет нелинейности.

- Подходит для случаев, когда прямого сходства между векторами недостаточно.



In [ ]:
import os
import random

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt
from IPython.display import clear_output


#### Dot product attention (пример реализации)

Рассмотрим единственное состояние энкодера – вектор с размерностью `(n_hidden, 1)`, где `n_hidden = 3` - размерность вектора, который определяет единственное состояние энкодера:

In [ ]:
decoder_hidden_state = np.array([7, 11, 4]).astype(float)[:, None]

plt.figure(figsize=(2, 5))
plt.pcolormesh(decoder_hidden_state)
plt.colorbar()
plt.title('Decoder state')

In [ ]:
single_encoder_hidden_state = np.array([1, 5, 11]).astype(float)[:, None]

plt.figure(figsize=(2, 5))
plt.pcolormesh(single_encoder_hidden_state)
plt.colorbar()

In [ ]:
# вектор внимания как скалярное произведение
np.dot(decoder_hidden_state.T, single_encoder_hidden_state)

Обобщим на случай, когда у энкодера несколько скрытых состояний:

In [ ]:
encoder_hidden_states = np.array([
    [1, 5, 11],
    [7, 4, 1],
    [8, 12, 2],
    [-9, 0, 1]

]).astype(float).T

encoder_hidden_states

In [ ]:
# функция подсчета скалярных произведений
def dot_product_attention_score(decoder_hidden_state, encoder_hidden_states):
    '''
    decoder_hidden_state: np.array of shape (n_features, 1)
    encoder_hidden_states: np.array of shape (n_features, n_states)

    return: np.array of shape (1, n_states)
        Array with dot product attention scores
    '''
    attention_scores = np.dot(decoder_hidden_state.T, encoder_hidden_states)
    return attention_scores

In [ ]:
# вектор скрытого состояния
dot_product_attention_score(decoder_hidden_state, encoder_hidden_states)

In [ ]:
#
def softmax(vector):
    '''
    vector: np.array of shape (n, m)

    return: np.array of shape (n, m)
        Matrix where softmax is computed for every row independently
    '''
    nice_vector = vector - vector.max()
    exp_vector = np.exp(nice_vector)
    exp_denominator = np.sum(exp_vector, axis=1)[:, np.newaxis]
    softmax_ = exp_vector / exp_denominator
    return softmax_

In [ ]:
# вектор весов всех состояний
weights_vector = softmax(dot_product_attention_score(decoder_hidden_state, encoder_hidden_states))

weights_vector

**Итоговый вектор внимания** представляет собой взвешенную сумму всех состояний энкодера, где веса определяются релевантностью каждого состояния энкодера текущему состоянию декодера.


In [ ]:
def dot_product_attention(decoder_hidden_state, encoder_hidden_states):
    '''
    decoder_hidden_state: np.array of shape (n_features, 1)
    encoder_hidden_states: np.array of shape (n_features, n_states)

    return: np.array of shape (n_features, 1)
        Final attention vector
    '''
    softmax_vector = softmax(dot_product_attention_score(decoder_hidden_state, encoder_hidden_states))
    attention_vector = softmax_vector.dot(encoder_hidden_states.T).T
    return attention_vector

**1)** Реализуйте мультипликативное внимание

In [ ]:
encoder_hidden_states_complex = np.array([
    [1, 5, 11, 4, -4],
    [7, 4, 1, 2, 2],
    [8, 12, 2, 11, 5],
    [-9, 0, 1, 8, 12]

]).astype(float).T

# Дана матрица весов
W_mult = np.array([
    [-0.78, -0.97, -1.09, -1.79,  0.24],
    [ 0.04, -0.27, -0.98, -0.49,  0.52],
    [ 1.08,  0.91, -0.99,  2.04, -0.15]
])

In [ ]:
def multiplicative_attention(decoder_hidden_state, encoder_hidden_states, W_mult):
    '''
    decoder_hidden_state: np.array of shape (n_features_dec, 1)
    encoder_hidden_states: np.array of shape (n_features_enc, n_states)
    W_mult: np.array of shape (n_features_dec, n_features_enc)

    return: np.array of shape (n_features_enc, 1)
        Final attention vector
    '''

    W_enc = np.dot(W_mult, encoder_hidden_states)


    attention_scores = np.dot(decoder_hidden_state.T, W_enc)


    attention_weights = softmax(attention_scores)


    attention_vector = np.dot(encoder_hidden_states, attention_weights.T)

    return attention_vector

#### Сохраните результат

In [ ]:

attention_vector = multiplicative_attention(decoder_hidden_state, encoder_hidden_states_complex, W_mult)


np.save('nlp02.npy', attention_vector, allow_pickle=True)
print('Ответ сохранен в файл `nlp02.npy`')